In [1]:
from simulation import citygraph_dataset
# from learning import inductive_route_learning, eval_route_generator, bee_colony
from learning.bee_colony import main as main_bee  # я так обозвал
from learning.eval_route_generator import main as main_eval # я так обозвал
from omegaconf import OmegaConf, DictConfig
from simulation import drawing

from tqdm import tqdm
from pathlib import Path

In [ ]:
dataset = citygraph_dataset.DynamicCityGraphDataset(
    min_nodes=25,
    max_nodes=25,
    edge_keep_prob=0.7,
    data_type=citygraph_dataset.MIXED,  # or any other type you want
    directed=False,
    fully_connected_demand=True,  # default SIDE_LENGTH_M
    mumford_style=True,
    pos_only=False
)

# Generate graphs
n_graphs = 1000  # number of graphs you want to generate
graphs = [dataset.generate_graph(draw=False) for _ in tqdm(range(n_graphs))]

In [3]:
# import pickle
# from pathlib import Path

# # Путь для сохранения
# save_path = Path('./output_graphs')
# if not save_path.exists():
#     save_path.mkdir(parents=True)

# # Сохраняем объект в файл
# with open(save_path / 'raw_graphs_1000.pkl', 'wb') as ff:
#     pickle.dump(graphs, ff)


In [4]:
# import pickle

# with open(save_path / 'raw_graphs_1000.pkl', 'rb') as f:
#     graphs = pickle.load(f)

In [5]:
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
import os

cfg_dir = os.path.abspath("../TNDP_learning/cfg")

### Обучение LC

In [6]:
# with initialize_config_dir(config_dir=cfg_dir, job_name="app"):
#     cfg_learn = compose(config_name="ppo_20nodes_copy.yaml", # в этот файл можно положить путь к пикл файлу с графами, также можно попробовать другие параметры на обучение
#                             overrides=["+run_name=random_graphs_weighted_connectivity"]) # при желании можно накинуть ему имя 

In [7]:
# print(OmegaConf.to_yaml(cfg_learn)) # чисто проверка как выглядит конфиг 

In [8]:
# from learning import inductive_route_learning
# inductive_route_learning.setup_and_train(cfg_learn) # веса для модели вернутся в папку output, там будет файл .pt / при желании можно накинуть ему имя 

### Стартовый набор маршрутов

In [9]:
dataset_name = 'vologda'
demand_time_weight = 0.33
route_time_weight = 0.33
median_connectivity_weight = 0.33
experiment_name = f'exp_weighted_connectivity_{dataset_name}_pp_{demand_time_weight}_op_{route_time_weight}_cp_{median_connectivity_weight}'
initial_routes_name = experiment_name+ '_starting'
generated_routes_name = experiment_name + '_generated'

In [10]:
with initialize_config_dir(config_dir=cfg_dir, version_base=None):
    
    cfg_eval = compose(
        config_name="eval_model_mumford",   # из @hydra.main
        overrides=[
            f"+eval={dataset_name}", # конфиг в котором задается кол-во маршрутов и их макс и мин длины (можно заменить на vo для теста по ваське)
            "+model.weights=../TNDP_learning/output/inductive_random_graphs_weighted_connectivity_checkpoints/iter990.pt", # путь к весам модели
            f"++run_name={initial_routes_name}", # имя запуска, вернет pickle файл с тензором в output_routes
            f"++experiment.cost_function.kwargs.demand_time_weight={demand_time_weight}",
            f"++experiment.cost_function.kwargs.route_time_weight={route_time_weight}",
            f"++experiment.cost_function.kwargs.median_connectivity_weight={median_connectivity_weight}"
        ]
    )

In [11]:
print(OmegaConf.to_yaml(cfg_eval)) # чисто проверка как выглядит конфиг 

experiment:
  logdir: null
  anomaly: false
  cpu: false
  seed: 0
  symmetric_routes: true
  cost_function:
    type: mine
    kwargs:
      mean_stop_time_s: 0
      avg_transfer_wait_time_s: 300
      demand_time_weight: 0.33
      route_time_weight: 0.33
      median_connectivity_weight: 0.33
      constraint_violation_weight: 5.0
      variable_weights: true
      pp_fraction: 0.15
      op_fraction: 0.15
      mcw_fraction: 0.15
model:
  common:
    dropout: 0.0
    nonlin_type: ReLU
    embed_dim: 64
  route_generator:
    kwargs:
      force_linking_unlinked: false
      logit_clip: null
      n_nodepair_layers: 3
      n_pathscorer_layers: 3
      pathscorer_hidden_dim: 16
      n_halt_layers: 3
      halt_scorer_type: endpoints
      serial_halting: true
    type: PathCombiningRouteGenerator
  backbone_gn:
    net_type: graph attn
    kwargs:
      n_layers: 5
      in_node_dim: 4
      in_edge_dim: 14
      use_norm: false
      n_heads: 4
      dense: false
  weights: ../TN

In [12]:
metrics, unserved_demand = main_eval(cfg_eval)

KeyboardInterrupt: 

### Генерация 

In [16]:
with initialize_config_dir(config_dir=cfg_dir, version_base=None):
    cfg_neural = compose(
        config_name="neural_bco_mumford",
        overrides=[
            f"+eval={dataset_name}", # конфиг в котором задается кол-во маршрутов и их макс и мин длины (можно заменить на vo для теста по ваське)
            "+model.weights=../TNDP_learning/output/inductive_random_graphs_checkpoints/iter990.pt", # путь к весам модели
            f"++run_name={generated_routes_name}", # имя запуска, вернет pickle файл с тензором в output_routes
            f"++experiment.cost_function.kwargs.demand_time_weight={demand_time_weight}",
            f"++experiment.cost_function.kwargs.route_time_weight={route_time_weight}",
            f"++experiment.cost_function.kwargs.median_connectivity_weight={median_connectivity_weight}",
            f"init.path=output_routes/vologda_routes.pkl",
        ]
    )

In [ ]:
# print(OmegaConf.to_yaml(cfg_neural)) # чисто проверка как выглядит конфиг 

In [17]:
metrics, unserved_demand = main_bee(cfg_neural)

KeyboardInterrupt: 

In [ ]:
metrics['median_connectivity'] /= 60

In [ ]:
# Порядок нужных ключей
keys_order = ['ATT', 'RTT', 'median_connectivity','median_connectivity_weighted', 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']

print('\t'.join(str(round(metrics[key].item(),3)) for key in keys_order))
print('\t'.join(str(round(metrics[key].item(),3)) for key in keys_order))

10.954	244.0	12.0	0.332	0.0	84.393	13.166	2.441
10.954	244.0	12.0	0.332	0.0	84.393	13.166	2.441


In [ ]:
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
from learning.eval_route_generator import main as main_eval
from learning.bee_colony import main as main_bee
from tqdm import tqdm

# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()

# Все параметры экспериментов
experiments = [
    ("mandl", 1, 0, 0),
    ("mandl", 0, 1, 0),
    ("mandl", 0, 0, 1),
    ("mandl", 0.5, 0.5, 0),
    ("mandl", 0.5, 0, 0.5),
    ("mandl", 0, 0.5, 0.5),
    ("mandl", 0.33, 0.33, 0.33),
    ("mumford0", 1, 0, 0),
    ("mumford0", 0, 1, 0),
    ("mumford0", 0, 0, 1),
    ("mumford0", 0.5, 0.5, 0),
    ("mumford0", 0.5, 0, 0.5),
    ("mumford0", 0, 0.5, 0.5),
    ("mumford0", 0.33, 0.33, 0.33),
    ("mumford1", 1, 0, 0),
    ("mumford1", 0, 1, 0),
    ("mumford1", 0, 0, 1),
    ("mumford1", 0.5, 0.5, 0),
    ("mumford1", 0.5, 0, 0.5),
    ("mumford1", 0, 0.5, 0.5),
    ("mumford1", 0.33, 0.33, 0.33),
    ("mumford2", 1, 0, 0),
    ("mumford2", 0, 1, 0),
    ("mumford2", 0, 0, 1),
    ("mumford2", 0.5, 0.5, 0),
    ("mumford2", 0.5, 0, 0.5),
    ("mumford2", 0, 0.5, 0.5),
    ("mumford2", 0.33, 0.33, 0.33),
    ("mumford3", 1, 0, 0),
    ("mumford3", 0, 1, 0),
    ("mumford3", 0, 0, 1),
    ("mumford3", 0.5, 0.5, 0),
    ("mumford3", 0.5, 0, 0.5),
    ("mumford3", 0, 0.5, 0.5),  
    ("mumford3", 0.33, 0.33, 0.33),
]

# Путь к весам модели
model_weights_path = "../TNDP_learning/output/inductive_random_graphs_weighted_connectivity_checkpoints/iter990.pt"

# CSV файл для результатов
results_file = Path("experiment_results.csv")
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow([
            "dataset", "demand_time", "route_time", "connectivity",
            "ATT", "RTT", "median_connectivity","median_connectivity_weighted", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

# Запуск экспериментов с tqdm
for dataset_name, dt, rt, ct in tqdm(experiments, desc="Running experiments"):
    try:
        experiment_name = f"exp_weighted_connectivity_{dataset_name}_pp_{dt}_op_{rt}_cp_{ct}"
        initial_routes_name = experiment_name + '_starting'
        generated_routes_name = experiment_name + '_generated'

        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_eval = compose(
                config_name="eval_model_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={initial_routes_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}"
                ]
            )
        eval_metrics, unserved_demand = main_eval(cfg_eval)

        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_bee = compose(
                config_name="neural_bco_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={generated_routes_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                    f"init.path=output_routes/nn_construction_{initial_routes_name}_routes.pkl",
                ]
            )
        bee_metrics, unserved_demand = main_bee(cfg_bee)
        bee_metrics['median_connectivity'] /= 60

        keys_order = ['ATT', 'RTT', 'median_connectivity', "median_connectivity_weighted", 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
        row = [dataset_name.lower(), dt, rt, ct] + [round(bee_metrics[k].item(), 4) for k in keys_order]

        with open(results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)

    except Exception as e:
        print(f"[✗] Failed {experiment_name}: {e}")

In [ ]:
# Эксперименты только с mandl
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from learning.eval_route_generator import main as main_eval
from tqdm import tqdm
# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()

model_weights_path = "../TNDP_learning/output/inductive_random_graphs_weighted_connectivity_checkpoints/iter990.pt"

mandl_experiments = [
    ("mandl", 1, 0, 0),
    ("mandl", 0, 1, 0),
    ("mandl", 0, 0, 1),
    ("mandl", 0.5, 0.5, 0),
    ("mandl", 0.5, 0, 0.5),
    ("mandl", 0, 0.5, 0.5),
    ("mandl", 0.33, 0.33, 0.33),
]

# CSV файл
results_file = Path("eval_only_mandl.csv")
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        csv.writer(f).writerow([
            "dataset", "demand_time", "route_time", "connectivity",
            "ATT", "RTT", "median_connectivity", "median_connectivity_weighted", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

# Запуск экспериментов только main_eval
for dataset_name, dt, rt, ct in tqdm(mandl_experiments, desc="Evaluating mandl"):
    try:
        run_name = f"mandl_weighted_connectivity_eval_pp_{dt}_op_{rt}_cp_{ct}"

        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_eval = compose(
                config_name="eval_model_mumford",
                overrides=[
                    f"+eval={dataset_name}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={run_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                ]
            )

        metrics = main_eval(cfg_eval)
        keys_order = ['ATT', 'RTT', 'median_connectivity', "median_connectivity_weighted", 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
        row = [dataset_name, dt, rt, ct] + [round(metrics[k].item(), 4) for k in keys_order]

        with open(results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)

    except Exception as e:
        print(f"[✗] Failed eval for {run_name}: {e}")

In [ ]:
# Эксперименты только с mandl
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from learning.eval_route_generator import main as main_eval
from tqdm import tqdm
# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()

model_weights_path = "../TNDP_learning/output/inductive_random_graphs.pt"

mandl_experiments = [
    ("mumford1", 0.0, 0.5, 0.0),
    ("mumford1", 0.2, 0.5, 0.0),
    ("mumford1", 0.4, 0.5, 0.0),
    ("mumford1", 0.6, 0.5, 0.0),
    ("mumford1", 0.8, 0.5, 0.0),
    ("mumford1", 1.0, 0.5, 0.0),
]


# CSV файл
results_file = Path("var_pass_coef.csv")
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        csv.writer(f).writerow([
            "dataset", "demand_time", "route_time", "connectivity", "seed",
            "ATT", "RTT", "median_connectivity", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

# Запуск экспериментов только main_eval
for dataset_name, dt, rt, ct in tqdm(mandl_experiments, desc="Evaluating mandl"):
    for seed in range (0, 10):
        try:
            run_name = f"mandl_eval_pp_{dt}_op_{rt}_cp_{ct}"

            with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
                cfg_eval = compose(
                    config_name="eval_model_mumford",
                    overrides=[
                        f"+eval={dataset_name}",
                        f"++eval.n_routes=5",
                        f"+model.weights={model_weights_path}",
                        f"++run_name={run_name}",
                        f"++experiment.seed={seed}",
                        f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                        f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                        f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                    ]
                )

            metrics, unserved_demand = main_eval(cfg_eval)
            keys_order = ['ATT', 'RTT', 'median_connectivity', 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
            row = [dataset_name, dt, rt, ct, seed] + [round(metrics[k].item(), 4) for k in keys_order]

            with open(results_file, mode='a', newline='') as f:
                csv.writer(f).writerow(row)

        except Exception as e:
            print(f"[✗] Failed eval for {run_name}: {e}")

Evaluating mandl: 100%|██████████| 6/6 [19:45:09<00:00, 11851.58s/it]  


In [ ]:
import csv
from pathlib import Path
from hydra import initialize_config_dir, compose
from learning.eval_route_generator import main as main_eval

# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()
model_weights_path = "../TNDP_learning/output/inductive_random_graphs.pt"

# Два эксперимента
experiments = [
    ("mumford1", 1.0, 0.5, 0.0, "exp_dt1_rt05_ct0"),
    ("mumford1", 0.0, 0.5, 1.0, "exp_dt0_rt05_ct1"),
]

# CSV файл
results_file = Path("small.csv")
if not results_file.exists():
    with open(results_file, mode='w', newline='') as f:
        csv.writer(f).writerow([
            "dataset", "demand_time", "route_time", "connectivity", "seed",
            "ATT", "RTT", "median_connectivity", "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
        ])

# Запуск экспериментов
for dataset_name, dt, rt, ct, run_name in experiments:
    try:
        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_eval = compose(
                config_name="eval_model_mumford",
                overrides=[
                    f"+eval={dataset_name}",
                    f"++eval.n_routes=5",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={run_name}",
                    f"++experiment.seed=0",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                ]
            )

        metrics, unserved_demand = main_eval(cfg_eval)
        keys_order = ['ATT', 'RTT', 'median_connectivity', 'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']
        row = [dataset_name, dt, rt, ct, 0] + [round(metrics[k].item(), 4) for k in keys_order]

        with open(results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)

        print(f"[✓] Finished {run_name}")

    except Exception as e:
        print(f"[✗] Failed eval for {run_name}: {e}")


/root/TNDP_learning/learning/utils.py:321: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand,  metrics)


[✓] Finished exp_dt1_rt05_ct0
[✓] Finished exp_dt0_rt05_ct1


In [3]:
import csv
import torch
from pathlib import Path
from hydra import initialize_config_dir, compose
from omegaconf import OmegaConf
from learning.eval_route_generator import main as main_eval
from learning.bee_colony import main as main_bee
from tqdm import tqdm

# Путь к конфигам
cfg_dir = Path("../TNDP_learning/cfg").resolve()

# Все параметры экспериментов
experiments = [
    ("mandl", 1, 0, 0),
    ("mandl", 0, 1, 0),
    ("mandl", 0, 0, 1),
    ("mandl", 0.5, 0.5, 0),
    ("mandl", 0.5, 0, 0.5),
    ("mandl", 0, 0.5, 0.5),
    ("mandl", 0.33, 0.33, 0.33),
    ("mumford0", 1, 0, 0),
    ("mumford0", 0, 1, 0),
    ("mumford0", 0, 0, 1),
    ("mumford0", 0.5, 0.5, 0),
    ("mumford0", 0.5, 0, 0.5),
    ("mumford0", 0, 0.5, 0.5),
    ("mumford0", 0.33, 0.33, 0.33),
    ("mumford1", 1, 0, 0),
    ("mumford1", 0, 1, 0),
    ("mumford1", 0, 0, 1),
    ("mumford1", 0.5, 0.5, 0),
    ("mumford1", 0.5, 0, 0.5),
    ("mumford1", 0, 0.5, 0.5),
    ("mumford1", 0.33, 0.33, 0.33),
    ("mumford2", 1, 0, 0),
    ("mumford2", 0, 1, 0),
    ("mumford2", 0, 0, 1),
    ("mumford2", 0.5, 0.5, 0),
    ("mumford2", 0.5, 0, 0.5),
    ("mumford2", 0, 0.5, 0.5),
    ("mumford2", 0.33, 0.33, 0.33),
    ("mumford3", 1, 0, 0),
    ("mumford3", 0, 1, 0),
    ("mumford3", 0, 0, 1),
    ("mumford3", 0.5, 0.5, 0),
    ("mumford3", 0.5, 0, 0.5),
    ("mumford3", 0, 0.5, 0.5),  
    ("mumford3", 0.33, 0.33, 0.33),
]

# Путь к весам модели
model_weights_path = str(Path("../TNDP_learning/output/inductive_random_graphs_weighted_connectivity.pt").resolve())

# Создание директорий для результатов
results_dir = Path("experiment_results")
results_dir.mkdir(exist_ok=True)

unserved_demand_dir = results_dir / "unserved_demand_matrices"
unserved_demand_dir.mkdir(exist_ok=True)

# CSV файлы для результатов каждой модели
lc_results_file = results_dir / "lc_results.csv"
neurobco_results_file = results_dir / "neurobco_results.csv"
bco_results_file = results_dir / "bco_results.csv"

# Инициализация CSV файлов
csv_header = [
    "dataset", "demand_time", "route_time", "connectivity",
    "ATT", "RTT", "median_connectivity", "median_connectivity_weighted", 
    "cost", "$d_{un}$", "$d_0$", "$d_1$", "$d_2$"
]

for results_file in [lc_results_file, neurobco_results_file, bco_results_file]:
    if not results_file.exists():
        with open(results_file, mode='w', newline='') as f:
            writer = csv.writer(f)
            writer.writerow(csv_header)

keys_order = ['ATT', 'RTT', 'median_connectivity', "median_connectivity_weighted", 
             'cost', '$d_{un}$', '$d_0$', '$d_1$', '$d_2$']

# Запуск экспериментов с tqdm
for dataset_name, dt, rt, ct in tqdm(experiments, desc="Running experiments"):
    param_name = f"dt{dt}_rt{rt}_ct{ct}"
    experiment_name = f"{dataset_name}_{param_name}"
    
    print(f"\n{'='*60}")
    print(f"Processing: {experiment_name}")
    print(f"{'='*60}")
    
    try:
        # ==================== LC (eval_model_mumford) ====================
        print(f"[1/3] Running LC...")
        initial_routes_name = f"LC_{dataset_name}_{param_name}_starting"
        
        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_lc = compose(
                config_name="eval_model_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={initial_routes_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}"
                ]
            )
        
        lc_metrics, lc_unserved_demand = main_eval(cfg_lc)
        
        # Сохранение unserved demand матрицы для LC
        unserved_demand_path = unserved_demand_dir / f"LC_{experiment_name}_unserved_demand.pt"
        torch.save(lc_unserved_demand, unserved_demand_path)
        
        row = [dataset_name.lower(), dt, rt, ct] + [
            round(lc_metrics[k].item(), 4) if k in lc_metrics else 0 for k in keys_order
        ]
        
        with open(lc_results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)
        
        print(f"[✓] LC completed")

    except Exception as e:
        print(f"[✗] LC failed: {e}")
        continue

    try:
        # ==================== NeuroBCO (neural_bco_mumford) ====================
        print(f"[2/3] Running NeuroBCO...")
        neuro_generated_name = f"NeuroBCO_{dataset_name}_{param_name}_generated"
        
        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_neuro = compose(
                config_name="neural_bco_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"+model.weights={model_weights_path}",
                    f"++run_name={neuro_generated_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                    f"init.path=output_routes/nn_construction_{initial_routes_name}_routes.pkl",
                ]
            )
        
        neuro_metrics, neuro_unserved_demand = main_bee(cfg_neuro)
        neuro_metrics['median_connectivity'] /= 60
        
        # Сохранение unserved demand матрицы для NeuroBCO
        unserved_demand_path = unserved_demand_dir / f"NeuroBCO_{experiment_name}_unserved_demand.pt"
        torch.save(neuro_unserved_demand, unserved_demand_path)
        
        row = [dataset_name.lower(), dt, rt, ct] + [
            round(neuro_metrics[k].item(), 4) for k in keys_order
        ]
        
        with open(neurobco_results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)
        
        print(f"[✓] NeuroBCO completed")

    except Exception as e:
        print(f"[✗] NeuroBCO failed: {e}")

    try:
        # ==================== BCO (bco_mumford) ====================
        print(f"[3/3] Running BCO...")
        bco_generated_name = f"BCO_{dataset_name}_{param_name}_generated"
        
        with initialize_config_dir(config_dir=str(cfg_dir), version_base=None):
            cfg_bco = compose(
                config_name="bco_mumford",
                overrides=[
                    f"+eval={dataset_name.lower()}",
                    f"++run_name={bco_generated_name}",
                    f"++experiment.cost_function.kwargs.demand_time_weight={dt}",
                    f"++experiment.cost_function.kwargs.route_time_weight={rt}",
                    f"++experiment.cost_function.kwargs.median_connectivity_weight={ct}",
                    f"init.path=output_routes/nn_construction_{initial_routes_name}_routes.pkl",
                ]
            )
        
        bco_metrics, bco_unserved_demand = main_bee(cfg_bco)
        bco_metrics['median_connectivity'] /= 60
        
        # Сохранение unserved demand матрицы для BCO
        unserved_demand_path = unserved_demand_dir / f"BCO_{experiment_name}_unserved_demand.pt"
        torch.save(bco_unserved_demand, unserved_demand_path)
        
        row = [dataset_name.lower(), dt, rt, ct] + [
            round(bco_metrics[k].item(), 4) for k in keys_order
        ]
        
        with open(bco_results_file, mode='a', newline='') as f:
            csv.writer(f).writerow(row)
        
        print(f"[✓] BCO completed")

    except Exception as e:
        print(f"[✗] BCO failed: {e}")
    
    print(f"[✓] Completed all models for {experiment_name}")

print("\n" + "="*60)
print("=== All experiments completed ===")
print("="*60)
print(f"Results directory: {results_dir}")
print(f"  LC results: {lc_results_file}")
print(f"  NeuroBCO results: {neurobco_results_file}")
print(f"  BCO results: {bco_results_file}")
print(f"  Unserved demand matrices: {unserved_demand_dir}/")

Running experiments:   0%|          | 0/35 [00:00<?, ?it/s]


Processing: mandl_dt1_rt0_ct0
[1/3] Running LC...
[✓] LC completed
[2/3] Running NeuroBCO...
[✓] NeuroBCO completed
[3/3] Running BCO...


Running experiments:   3%|▎         | 1/35 [00:17<10:00, 17.65s/it]

[✓] BCO completed
[✓] Completed all models for mandl_dt1_rt0_ct0

Processing: mandl_dt0_rt1_ct0
[1/3] Running LC...
[✓] LC completed
[2/3] Running NeuroBCO...
[✓] NeuroBCO completed
[3/3] Running BCO...


Running experiments:   6%|▌         | 2/35 [00:32<08:44, 15.88s/it]

[✓] BCO completed
[✓] Completed all models for mandl_dt0_rt1_ct0

Processing: mandl_dt0_rt0_ct1
[1/3] Running LC...
[✓] LC completed
[2/3] Running NeuroBCO...


Running experiments:   6%|▌         | 2/35 [00:38<10:36, 19.28s/it]


KeyboardInterrupt: 

Running routes_length experiments:   0%|          | 0/6 [00:00<?, ?it/s]


EXPERIMENT: routes_length_MUMFORD2_pp0.0_op0.0_cp1.0_routes26
[1/2] LC → 26 routes


/root/TNDP_learning/learning/utils.py:332: UserWarning: std(): degrees of freedom is <= 0. Correction should be strictly less than the reduction factor (input numel divided by output numel). (Triggered internally at /pytorch/aten/src/ATen/native/ReduceOps.cpp:1831.)
  out_stats = (final_costs.mean(), final_costs.std(), unserved_demand, all_metrics)


[Success] LC done
[2/2] NeuroBCO (LC init)


Running routes_length experiments:  17%|█▋        | 1/6 [01:05<05:26, 65.30s/it]

[Success] NeuroBCO done 
[Completed] routes_length_MUMFORD2_pp0.0_op0.0_cp1.0_routes26


EXPERIMENT: routes_length_MUMFORD2_pp0.0_op0.0_cp1.0_routes36
[1/2] LC → 36 routes
[Success] LC done
[2/2] NeuroBCO (LC init)


Running routes_length experiments:  17%|█▋        | 1/6 [01:44<08:42, 104.40s/it]


KeyboardInterrupt: 